# 05 · Benchmark model — the floor the real model has to clear
**AAI-540 · Group 4 · Criteo CTR**

Before any tuned model, two deliberately simple baselines are fitted on the
same training split and scored on the same held-out test split:

| Benchmark | What it does | Why it is here |
|---|---|---|
| `global_ctr` | predicts the training click rate for every impression | the heuristic. Perfectly calibrated in aggregate, so it scores a *respectable log loss* while being useless for ranking (AUC 0.5) |
| `two_feature_logistic` | L2 logistic regression on the two highest-signal features, one per source column | shows how much of the achievable lift comes from two of the thirty-nine columns |

**Why two baselines and not one.** They fail in different ways, and seeing both
fail differently is what makes the comparison informative. If we reported only
log loss against `global_ctr`, a model with no discriminative power at all
could look acceptable. If we reported only AUC, a model that ranks well but
mis-scales its probabilities would look fine — and a bidding system multiplies
that probability by a value, so the mis-scaling is what costs money.

The test log loss of `global_ctr` becomes the **quality-gate threshold** the
CI/CD pipeline uses in notebook 10. A candidate that cannot beat it is not
registered.

All model code lives in `src/criteo_ctr/benchmark.py` and the metrics in
`src/criteo_ctr/evaluation.py`; both are unit-tested (`tests/test_benchmark.py`,
`tests/test_evaluation.py`), so the numbers below come from the same functions
the pipeline gate and the production monitors use.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import numpy as np
import pandas as pd
import sagemaker

from criteo_ctr import config as C
from criteo_ctr import benchmark as B
from criteo_ctr import evaluation as E
from criteo_ctr import features as F
from criteo_ctr.io_utils import Store, s3_uri, canonical_columns

sess = sagemaker.Session()
boto_sess = sess.boto_session
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)

FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
ART_DIR = os.path.abspath("../artifacts"); os.makedirs(ART_DIR, exist_ok=True)

print(f"region={region}\nbucket={bucket}\nsplits={s3_uri(bucket, C.SPLITS_PREFIX)}")

## 1 · Load the splits written by notebook 04

Read from the Parquet copies rather than re-deriving anything: these are the
exact rows the Feature Store join produced, with the categorical encoder
already fitted on the training split only. Re-deriving features here would
create a second implementation that could drift from the one the pipeline uses.

In [ ]:
splits = {}
for name in C.SPLIT_ORDER:
    frame = store.read_parquet_prefix(f"{C.SPLITS_PREFIX}/parquet/split={name}/")
    splits[name] = canonical_columns(frame)
    print(f"{name:11s} {len(splits[name]):>9,} rows   CTR {splits[name][C.LABEL].mean():.5f}")

manifest = json.loads(
    boto_sess.client("s3").get_object(
        Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/split_manifest.json")["Body"].read())
assert manifest["encoder_fitted_on"] == "train", "splits were not built train-only"
feature_cols = F.model_feature_columns()
print(f"\n{len(feature_cols)} model features; encoder fitted on "
      f"{manifest['encoder_n_train']:,} training rows")

## 2 · Fit both benchmarks on the training split

In [ ]:
train, test = splits["train"], splits["test"]
y_test = test[C.LABEL].to_numpy()

t0 = time.time()
benchmarks = B.fit_benchmarks(train)
print(f"fitted in {time.time() - t0:.1f}s\n")

print(f"global_ctr           predicts {benchmarks['global_ctr'].rate_:.5f} for every row")
logistic = benchmarks["two_feature_logistic"]
print(f"two_feature_logistic features: {logistic.columns_}")
print(f"                     coefficients: "
      f"{np.round(logistic.coef_, 4).tolist()}, intercept {logistic.intercept_:.4f}")
print(f"                     train log loss {logistic.loss_history_[-1]:.5f} "
      f"(from {logistic.loss_history_[0]:.5f})")
print("\ntop univariate signal (|corr| with label, training split only):")
for col, score in list(logistic.feature_scores_.items())[:8]:
    print(f"  {col:14s} {score:.4f}")

## 3 · Score both on the held-out test split

The test split is scored once, here, and the number is not used to choose
anything — selection decisions all use validation. The default classification
threshold in `evaluate` is the **base rate**, not 0.5: at a single-digit click
rate, a 0.5 cut-off predicts "no click" for every impression and reports a
flattering accuracy with zero recall.

In [ ]:
reports = []
for name, model in benchmarks.items():
    probabilities = model.predict_proba(B.feature_frame(test))
    reports.append(E.evaluate(y_test, probabilities, name=name))

comparison = E.compare(reports)
table = pd.DataFrame(comparison["rows"]).set_index("name")
display(table[["log_loss", "normalised_entropy", "roc_auc", "pr_auc",
               "expected_calibration_error"]].round(5))

print(f"\nbase-rate log loss on test: {reports[0]['base_rate_log_loss']:.5f}")
print("normalised entropy < 1.0 means the model beats predicting the average click rate.")
for row in comparison["rows"]:
    verdict = "beats" if row["improves_on_benchmark"] else "does not beat"
    print(f"  {row['name']:22s} {verdict} the global-CTR heuristic")

## 4 · The point of the exercise, in one figure

Left: log loss — the constant predictor looks *almost reasonable*.
Right: AUC — it is at 0.5, i.e. it cannot rank at all. A gate on log loss alone
would have let a model with no discriminative value through, which is why
`evaluation.evaluate` reports both and the design document commits to both.

In [ ]:
import matplotlib.pyplot as plt

names = [r["name"] for r in reports]
colors = ["#b0b7c3", "#6a8caf"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
axes[0].bar(names, [r["log_loss"] for r in reports], color=colors)
axes[0].axhline(reports[0]["base_rate_log_loss"], ls="--", lw=1, color="#c0392b",
                label="base-rate log loss")
axes[0].set_title("Log loss (lower is better)"); axes[0].legend(fontsize=8)

axes[1].bar(names, [r["roc_auc"] for r in reports], color=colors)
axes[1].axhline(0.5, ls="--", lw=1, color="#c0392b", label="no ranking power")
axes[1].set_ylim(0.4, 1.0); axes[1].set_title("ROC-AUC"); axes[1].legend(fontsize=8)

for report, color in zip(reports, colors):
    rows = report["calibration"]
    axes[2].plot([r["mean_predicted"] for r in rows], [r["observed_ctr"] for r in rows],
                 "o-", color=color, label=report["name"], ms=4)
lims = [0, max(r["max_predicted"] for r in reports[-1]["calibration"]) * 1.1]
axes[2].plot(lims, lims, ls="--", lw=1, color="#555", label="perfect calibration")
axes[2].set_xlabel("mean predicted"); axes[2].set_ylabel("observed CTR")
axes[2].set_title("Calibration by score bin"); axes[2].legend(fontsize=8)

for ax in axes[:2]:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
path = os.path.join(FIG_DIR, "12_benchmarks.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/12_benchmarks.png")
plt.show()

## 5 · Lift by decile — the ad-operations view

"If we bid only on the top decile of predicted impressions, how much better
than average is the click rate we get?" A constant predictor has no deciles to
speak of; the two-feature model already separates traffic.

In [ ]:
lift = pd.DataFrame(reports[-1]["lift"]).set_index("decile")
display(lift[["count", "observed_ctr", "lift", "cumulative_positive_share"]].round(4))

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(lift.index, lift["lift"], color="#6a8caf")
ax.axhline(1.0, ls="--", lw=1, color="#c0392b", label="population average")
ax.set_xlabel("score decile (1 = highest predicted CTR)"); ax.set_ylabel("lift")
ax.set_title(f"Lift by decile — {reports[-1]['name']}"); ax.legend(fontsize=8)
plt.tight_layout()
path = os.path.join(FIG_DIR, "13_benchmark_lift.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/13_benchmark_lift.png")
plt.show()

## 6 · Publish the benchmark metrics and the pipeline gate threshold

`benchmark_metrics.json` is read by notebook 06 (to compare the trained model
against the same numbers) and by notebook 10 (which passes
`gate_threshold_log_loss` into the pipeline as `LogLossThreshold`). Writing it
once here is what keeps the three notebooks from disagreeing about what the
baseline was.

In [ ]:
payload = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "test_rows": int(len(test)),
    "test_ctr": float(y_test.mean()),
    "base_rate_log_loss": reports[0]["base_rate_log_loss"],
    "gate_threshold_log_loss": reports[0]["log_loss"],
    "comparison": comparison,
    "reports": {r["name"]: {k: v for k, v in r.items() if k != "name"} for r in reports},
    "models": {name: model.to_dict() for name, model in benchmarks.items()},
}
key = f"{C.ARTIFACTS_PREFIX}/benchmark_metrics.json"
store.put_json(payload, key)
with open(os.path.join(ART_DIR, "benchmark_metrics.json"), "w") as fh:
    json.dump(payload, fh, indent=2, default=str)

print(f"wrote s3://{bucket}/{key}")
print(f"\nCI/CD quality gate threshold (LogLossThreshold) = "
      f"{payload['gate_threshold_log_loss']:.6f}")
print("Any candidate in notebook 10 must score at or below this on the test split,\n"
      "or the pipeline takes the fail branch and nothing is registered.")

### What this notebook establishes

* A **minimum viable product** exists: a model that scores every impression,
  end to end, from the Feature Store splits.
* A **floor**: `global_ctr`'s test log loss, which becomes the CI/CD gate.
* A **reason to report two metrics**: the heuristic's respectable log loss
  beside its 0.5 AUC is the clearest argument in the project for why log loss
  alone is not a sufficient acceptance criterion.

Next: `06_model_training.ipynb` trains the XGBoost candidate, tunes it against
validation log loss, and compares it with these baselines on the same split.